# Качество воды в Эстонии — Полное руководство по ML-проекту

**Автор:** Anton Sokolov  
**Курс:** Masinõpe (TalTech, весна 2026)  
**Язык ноутбука:** русский (учебный материал)

---

## Что это за ноутбук?

Это **главный учебный документ** проекта. Он написан так, чтобы его мог понять человек, который только начинает изучать машинное обучение. Здесь:

- объясняется каждый термин, аббревиатура и метод
- показывается весь путь от сырых данных до работающей модели
- обсуждается, как проект можно развивать дальше (погода, спутниковые данные, временные ряды)

> **Важно:** Ноутбук — это не просто код. Это _рассуждение_. Читайте текст между клетками — там самое главное.

---

## Оглавление

1. Что такое машинное обучение (теория для новичков)
2. Постановка задачи — конкретно и честно
3. Данные: источник, структура, что означают поля
4. Загрузка данных
5. Первичный осмотр: форма, типы, пропуски
6. Целевая переменная: баланс классов
7. Разведочный анализ (EDA): время, география, параметры
8. Инженерия признаков (Feature Engineering)
9. Разделение на обучающую и тестовую выборки
10. Предобработка: заполнение пропусков, масштабирование
11. Модель 1: Логистическая регрессия (baseline)
12. Метрики: как понять, хороша ли модель
13. Модель 2: Случайный лес (Random Forest)
14. Сравнение моделей
15. Feature Importance: что влияет на качество воды?
16. Будущее проекта: внешние данные и более умные модели
17. Итог

### Как пользоваться кодом в этом ноутбуке (для новичков)

- Между ячейками уже есть **теория**; **кодовые ячейки** повторяют шаги пайплайна: загрузка → таблица → графики → признаки → обучение → сравнение моделей.
- Если термин неясен, откройте **[`results/INSIGHTS.md`](../results/INSIGHTS.md)** — там **глоссарий** (что такое train/test, recall, ROC-AUC и т.д.) и **таблица «что делает каждый блок кода»** для этого ноутбука.
- Запускайте сверху вниз: некоторые переменные (`df`, `X`, `y`) нужны следующим ячейкам.

---
## 1. Что такое машинное обучение?

### Простое объяснение

**Машинное обучение (ML, Machine Learning)** — это способ научить компьютер находить закономерности в данных, не программируя правила вручную.

Пример без ML:
```
ЕСЛИ E.coli > 500 → «нарушение»
ЕСЛИ энтерококки > 200 → «нарушение»
ИНАЧЕ → «норма»
```
Это работает, если правила простые и известны заранее. Но что если:
- параметров десятки?
- нормативы разные для разных типов воды?
- важна комбинация показателей, а не каждый по отдельности?

ML позволяет **обучить модель на примерах** — она сама найдёт, что отличает «норму» от «нарушения».

---

### Типы задач машинного обучения

| Тип | Что делаем | Пример |
|-----|-----------|--------|
| **Классификация** | Отнести объект к одному из классов | Хорошая вода / плохая вода |
| **Регрессия** | Предсказать числовое значение | Сколько E.coli будет завтра? |
| **Кластеризация** | Разбить данные на группы без меток | Найти «похожие» места купания |

Наша задача — **классификация**.

---

### Ключевые понятия

**Признаки (Features, X)** — входные данные модели. В нашем случае: уровень E.coli, pH, дата, уезд...

**Целевая переменная (Target, y)** — то, что мы хотим предсказать. В нашем случае: `compliant` (0 или 1).

**Обучающая выборка (Train set)** — данные, на которых модель учится. Модель видит и признаки, и ответы.

**Тестовая выборка (Test set)** — данные, на которых проверяем модель. Модель видела только признаки, ответы прячем до проверки.

**Переобучение (Overfitting)** — модель «зазубрила» обучающие данные, но плохо работает на новых. Как студент, который выучил ответы, но не понял материал.

**Недообучение (Underfitting)** — модель слишком простая и не уловила закономерности даже в обучающих данных.

---

### Почему бинарная классификация?

**Бинарная классификация** — частный случай классификации, где только два исхода:
- `1` — норма (проба соответствует нормативам)
- `0` — нарушение (хотя бы один параметр превышает норму)

Наша целевая переменная — `compliant` (от английского "compliant" = "соответствующий").

---
## 2. Постановка задачи

### Что мы предсказываем?

> **Вопрос к модели:** По химическим и биологическим показателям пробы воды — соответствует ли она нормативам безопасности?

### Почему это важно?

Эстония — страна у моря, с тысячами озёр и публичных мест для купания. Государство регулярно берёт пробы воды и публикует результаты. Но:
- Данные появляются с задержкой
- Не все места проверяются одинаково часто
- Хотелось бы **предсказывать** риски до получения результатов лаборатории

Личная мотивация: я люблю купаться. Если модель может предупредить — это полезно напрямую, не абстрактно.

---

### Цена ошибок — это не одинаково!

В классификации есть два типа ошибок:

```
           Предсказание модели
                ↓        ↓
           НОРМА      НАРУШЕНИЕ
Факт: НОРМА    OK      False Positive (FP)
Факт: НАРУШЕНИЕ  False Negative (FN)    OK
```

**False Positive (FP)** — «ложная тревога». Вода нормальная, но модель сказала «нарушение».
→ Последствие: место закрыли зря, люди недовольны.

**False Negative (FN)** — «пропущенная опасность». Вода опасная, но модель сказала «норма».
→ Последствие: люди купаются в воде с E.coli. **Это хуже!**

> **Приоритет модели:** минимизировать False Negatives. Лучше лишний раз закрыть пляж, чем пропустить опасность.

Это напрямую влияет на выбор **метрики оценки** (раздел 12).

---

### Текущий этап vs Будущее

| Сейчас | В будущем |
|--------|-----------|
| Предсказание по лабораторным данным | + погода (температура, осадки, ветер) |
| Только данные Terviseamet | + спутниковые снимки (цветение водорослей) |
| Статический анализ | + временные ряды (тренд за неделю) |
| Один домен (supluskohad) | + все 5 доменов данных |

---
## 3. Данные: источник и структура

### Источник

**Terviseamet** (Департамент здоровья Эстонии) публикует результаты лабораторных анализов воды на портале [vtiav.sm.ee](https://vtiav.sm.ee). Данные доступны в формате **XML** (eXtensible Markup Language — расширяемый язык разметки, текстовый формат для хранения структурированных данных).

### Домены данных

Данные разбиты на 5 доменов:

| Ключ | Тип воды | Описание |
|------|----------|----------|
| `supluskoha` | Места для купания | Морские пляжи, озёра, реки |
| `veevark` | Водопроводная вода | Краны, резервуары, системы водоснабжения |
| `basseinid` | Бассейны | Публичные бассейны, спа, аквапарки |
| `joogivesi` | Питьевые источники | Природные источники, колодцы |
| `mineraalvesi` | Минеральная вода | Природные минеральные воды, термальные |

Сейчас реализованы парсеры opendata для `supluskoha`, `veevark` и `basseinid`. Начнём с `supluskoha`.

---

### Что означает каждый параметр?

**Микробиологические параметры** (основные индикаторы загрязнения):

| Параметр | Расшифровка | Единицы | Норма (купание) |
|----------|-------------|---------|----------------|
| `e_coli` | Escherichia coli — кишечная палочка. Бактерия, обитающая в кишечнике теплокровных. Её наличие означает фекальное загрязнение | КОЕ/100 мл | ≤ 500 (внутр.) |
| `enterococci` | Enterococcus — кишечные энтерококки. Другой индикатор фекального загрязнения, более устойчив во внешней среде | КОЕ/100 мл | ≤ 200 (внутр.) |

> **КОЕ** = Колониеобразующие единицы. Мера количества живых бактерий: сколько бактериальных колоний выросло на питательной среде из 100 мл воды.

**Физико-химические параметры:**

| Параметр | Расшифровка | Норма |
|----------|-------------|-------|
| `ph` | pH — водородный показатель. Кислотность воды. pH = 7 — нейтральная. < 7 — кислая, > 7 — щелочная | 6.0–9.0 |
| `transparency` | Прозрачность воды (läbipaistvus). Измеряется в метрах: диск Секки опускают в воду, пока он не станет невидимым | — |
| `turbidity` | Мутность (hägusus). Измеряется в NTU (Nephelometric Turbidity Units — нефелометрические единицы мутности) | ≤ 4 NTU |

**Химические параметры** (для водопроводной воды veevark):

| Параметр | Расшифровка | Норма |
|----------|-------------|-------|
| `nitrates` | Нитраты (NO₃⁻). Поступают из удобрений, сельского хозяйства | ≤ 50 мг/л |
| `nitrites` | Нитриты (NO₂⁻). Продукт разложения нитратов, более токсичны | ≤ 0.5 мг/л |
| `ammonium` | Аммоний (NH₄⁺). Признак загрязнения органикой | ≤ 0.5 мг/л |
| `iron` | Железо (Fe). Природный компонент, избыток ухудшает вкус и цвет | ≤ 0.2 мг/л |
| `manganese` | Марганец (Mn). Природный компонент, при избытке — вред здоровью | ≤ 0.05 мг/л |

### Целевая переменная

В XML каждый параметр имеет поле `vastavus` (эст. = соответствие):
- `jah` (эст. = да) → параметр в норме
- `ei` (эст. = нет) → параметр превышает норму

Наше правило: **если хотя бы один `vastavus = ei`** → `compliant = 0` (нарушение).

In [ ]:
# ── Импорт библиотек ──────────────────────────────────────────────────────────
#
# sys.path нужен, чтобы Python нашёл наши модули в папке ../src
import sys
sys.path.insert(0, '../src')

# pandas — главная библиотека для работы с табличными данными.
# DataFrame — это таблица: строки = наблюдения, столбцы = признаки.
import pandas as pd

# numpy — математика с массивами. Используем для числовых операций.
import numpy as np

# matplotlib и seaborn — визуализация данных.
import matplotlib.pyplot as plt
import seaborn as sns

# sklearn — scikit-learn, главная ML-библиотека Python.
# Импортируем по мере необходимости в каждом разделе.

# Наши модули из src/
from data_loader import load_domain, load_all
from features import add_time_features, add_ratio_features, build_dataset, impute_and_scale
from evaluate import evaluate_model, plot_confusion_matrix, plot_feature_importance, compare_models

# ── Настройки отображения ─────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['font.size'] = 11
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.3f}'.format)

print('Все библиотеки загружены успешно!')

---
## 4. Загрузка данных

Функция `load_domain()` делает следующее:
1. Проверяет, есть ли кэшированный XML в `data/raw/supluskoha.xml`
2. Если нет — скачивает с сервера Terviseamet
3. Парсит XML и возвращает pandas DataFrame

**XML (eXtensible Markup Language)** — текстовый формат для хранения иерархических данных. Похож на HTML, но теги придумываем сами. Каждая проба воды — один тег `<uuring>` (эст. = исследование).

> **Первый запуск:** если файла нет в кэше, ноутбук скачает данные с интернета. Это может занять 10–60 секунд в зависимости от скорости соединения.

In [ ]:
# Загружаем данные для домена supluskohad (места для купания)
# use_cache=True означает: использовать сохранённый XML, если он есть
df_raw = load_domain('supluskoha', use_cache=True)

print(f'Загружено проб: {len(df_raw)}')
print(f'Период данных: {df_raw["sample_date"].min()} — {df_raw["sample_date"].max()}')
print(f'\nКолонки ({len(df_raw.columns)} штук):')
print(df_raw.columns.tolist())

---
## 5. Первичный осмотр данных

Прежде чем что-либо делать с данными, нужно их **внимательно рассмотреть**. Это называется **санитарная проверка данных** (sanity check). Цель — убедиться, что данные загрузились правильно и понять их структуру.

Типичные вопросы:
- Сколько строк и столбцов? (df.shape)
- Какой тип у каждого столбца? (int? float? object = строка?)
- Есть ли пропущенные значения? (NaN = Not a Number — специальный маркер пропуска в pandas)
- Какой диапазон значений у числовых столбцов?

In [ ]:
# df.shape — кортеж (строки, столбцы)
print(f'Размер датасета: {df_raw.shape[0]} строк × {df_raw.shape[1]} столбцов')
print()

# Первые 5 строк — посмотрим, как выглядят данные
df_raw.head(5)

In [ ]:
# df.info() — типы данных и количество не-пустых значений
# object = строка (str в pandas называется object)
# float64 = число с плавающей точкой (64-битное)
# datetime64 = дата и время
df_raw.info()

In [ ]:
# df.describe() — базовая статистика для числовых столбцов:
# count = количество не-пустых значений
# mean = среднее арифметическое
# std = стандартное отклонение (насколько значения «разбросаны» вокруг среднего)
# min/max = минимум и максимум
# 25%, 50%, 75% = квартили (25% — первая четверть данных ниже этого значения)
df_raw.describe()

---
## 5.1 Пропущенные значения

**NaN (Not a Number)** в pandas — специальный маркер отсутствующего значения. Причины пропусков:
- Параметр не измерялся в этой пробе (не все параметры проверяются всегда)
- Данные не были внесены в систему
- Параметр не применим для данного типа воды

> **Важно:** пропуск — это тоже информация! Если параметр не измерялся, это может быть значимым само по себе. Именно поэтому мы создадим признаки-индикаторы пропусков (столбцы `*_missing`).

In [ ]:
# Считаем долю пропусков для каждого столбца
missing_pct = df_raw.isnull().sum() / len(df_raw) * 100
missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 4))
missing_pct.plot(kind='bar', ax=ax, color='#e07b7b', edgecolor='white')
ax.set_title('Доля пропущенных значений по столбцам (%)', fontsize=13)
ax.set_ylabel('% пропусков')
ax.axhline(y=50, color='darkred', linestyle='--', linewidth=1.5, label='50% порог')
ax.legend()
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

print('\nДетали:')
for col, pct in missing_pct.items():
    status = '⚠️ МНОГО' if pct > 50 else ('🔸' if pct > 20 else '✓')
    print(f'  {status} {col}: {pct:.1f}%')

---
## 6. Целевая переменная: баланс классов

**Дисбаланс классов (class imbalance)** — когда один класс встречается намного реже другого. Это типично для задач обнаружения аномалий:
- Хорошая вода — норма (много)
- Плохая вода — редкость (мало)

Чем опасен дисбаланс?

Представьте: 95% проб — норма, 5% — нарушения. Наивная модель, которая **всегда говорит «норма»**, получит 95% accuracy! Но она абсолютно бесполезна — она пропустит **все** нарушения.

Поэтому accuracy (точность) — плохая метрика при дисбалансе. Мы будем использовать F1, Recall, ROC-AUC (объяснение в разделе 12).

**Стратегии борьбы с дисбалансом:**
- `class_weight='balanced'` — в sklearn автоматически увеличивает штраф за ошибки на редком классе
- **SMOTE** (Synthetic Minority Oversampling Technique) — генерирует синтетические примеры редкого класса
- Изменение порога классификации — вместо 0.5 использовать 0.3 для более «параноидальной» модели

In [ ]:
# Убираем строки без целевой переменной (compliant = None)
df = df_raw.dropna(subset=['compliant']).copy()
print(f'Строк с известным статусом: {len(df)} из {len(df_raw)} ({len(df)/len(df_raw)*100:.1f}%)')

# Распределение классов
counts = df['compliant'].value_counts().sort_index()
ratios = df['compliant'].value_counts(normalize=True).sort_index() * 100

print('\nРаспределение классов:')
print(f'  Норма     (1): {counts.get(1, 0):>6} проб  ({ratios.get(1, 0):.1f}%)')
print(f'  Нарушение (0): {counts.get(0, 0):>6} проб  ({ratios.get(0, 0):.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Bar chart
colors = {0: '#e07b7b', 1: '#5b9bd5'}
axes[0].bar(
    ['Нарушение (0)', 'Норма (1)'],
    [counts.get(0, 0), counts.get(1, 0)],
    color=['#e07b7b', '#5b9bd5'], edgecolor='white', linewidth=1.5
)
for i, (label, val) in enumerate(zip(['Нарушение (0)', 'Норма (1)'], [counts.get(0, 0), counts.get(1, 0)])):
    axes[0].text(i, val + max(counts)*0.01, f'{val}\n({val/len(df)*100:.1f}%)',
                 ha='center', va='bottom', fontsize=11)
axes[0].set_title('Количество проб по классам')
axes[0].set_ylabel('Количество')

# Pie chart
axes[1].pie(
    [counts.get(0, 0), counts.get(1, 0)],
    labels=['Нарушение (0)', 'Норма (1)'],
    colors=['#e07b7b', '#5b9bd5'],
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[1].set_title('Доля классов')

plt.suptitle('Баланс целевой переменной (compliant)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 7. Разведочный анализ данных (EDA)

**EDA (Exploratory Data Analysis)** — разведочный анализ данных. Этап, на котором мы **смотрим, думаем, задаём вопросы** — ещё до обучения моделей.

Цели EDA:
- Понять распределение данных
- Найти аномалии и выбросы (outliers)
- Обнаружить закономерности (паттерны)
- Сформулировать гипотезы для модели
- Понять, какие признаки потенциально полезны

### Вопросы, которые мы хотим ответить:
1. Как менялось качество воды со временем?
2. Есть ли сезонность (лето vs зима)?
3. Какие уезды имеют больше нарушений?
4. Какой параметр чаще всего нарушает норму?
5. Как выглядит распределение E.coli в нормальных vs нарушающих пробах?

In [ ]:
# Добавляем временные признаки: год, месяц, сезон
df_time = add_time_features(df)

print('Добавлены признаки:', ['month', 'year', 'season', 'is_summer'])
print(f'\nГоды в данных: {sorted(df_time["year"].dropna().unique().astype(int).tolist())}')
print(f'Месяцы в данных: {sorted(df_time["month"].dropna().unique().astype(int).tolist())}')

In [ ]:
# ── Динамика по годам ─────────────────────────────────────────────────────────
yearly = df_time.groupby('year')['compliant'].agg(
    total='count',
    violations=lambda x: (x == 0).sum()
).reset_index()
yearly['violation_rate'] = yearly['violations'] / yearly['total'] * 100
yearly = yearly.sort_values('year')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Объём данных по годам
axes[0].bar(yearly['year'].astype(int), yearly['total'], color='#5b9bd5', edgecolor='white')
axes[0].set_title('Количество проб по годам')
axes[0].set_xlabel('Год')
axes[0].set_ylabel('Количество проб')

# Доля нарушений по годам
axes[1].plot(yearly['year'], yearly['violation_rate'], marker='o', color='#e07b7b', linewidth=2)
axes[1].fill_between(yearly['year'], yearly['violation_rate'], alpha=0.2, color='#e07b7b')
axes[1].set_title('Доля нарушений по годам (%)')
axes[1].set_xlabel('Год')
axes[1].set_ylabel('% нарушений')
axes[1].axhline(y=yearly['violation_rate'].mean(), color='gray',
                linestyle='--', linewidth=1, label=f'Среднее: {yearly["violation_rate"].mean():.1f}%')
axes[1].legend()

plt.suptitle('Временная динамика качества воды', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Среднегодовой % нарушений: {yearly["violation_rate"].mean():.1f}%')
print(f'Лучший год: {yearly.loc[yearly["violation_rate"].idxmin(), "year"]:.0f} ({yearly["violation_rate"].min():.1f}%)')
print(f'Худший год:  {yearly.loc[yearly["violation_rate"].idxmax(), "year"]:.0f} ({yearly["violation_rate"].max():.1f}%)')

In [ ]:
# ── Сезонность ────────────────────────────────────────────────────────────────
# Для мест купания ожидаем данные преимущественно летом (купальный сезон).
# Вопрос: в каком месяце больше нарушений?

monthly = df_time.groupby('month')['compliant'].agg(
    total='count',
    violations=lambda x: (x == 0).sum()
).reset_index()
monthly['violation_rate'] = monthly['violations'] / monthly['total'] * 100
monthly = monthly.sort_values('month')

month_names = {1:'Янв', 2:'Фев', 3:'Мар', 4:'Апр', 5:'Май', 6:'Июн',
               7:'Июл', 8:'Авг', 9:'Сен', 10:'Окт', 11:'Ноя', 12:'Дек'}
monthly['month_name'] = monthly['month'].map(month_names)

mean_vr = monthly['violation_rate'].mean()
colors_m = ['#e07b7b' if v > mean_vr else '#5b9bd5' for v in monthly['violation_rate']]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(monthly['month_name'], monthly['total'], color='#5b9bd5', edgecolor='white')
axes[0].set_title('Количество проб по месяцам')
axes[0].set_xlabel('Месяц')

axes[1].bar(monthly['month_name'], monthly['violation_rate'],
            color=colors_m, edgecolor='white')
axes[1].axhline(y=mean_vr, color='gray', linestyle='--',
                label=f'Среднее: {mean_vr:.1f}%')
axes[1].set_title('% нарушений по месяцам (красный = выше среднего)')
axes[1].set_xlabel('Месяц')
axes[1].set_ylabel('% нарушений')
axes[1].legend()

plt.suptitle('Сезонность нарушений качества воды', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Почему сезонность важна для будущей модели?

Летом:
- Высокая температура воды → быстрее размножаются бактерии
- Много людей → больше загрязнений
- Активное цветение водорослей → цианобактерии (синезелёные водоросли)

**Погодные данные в будущем:** добавив температуру воздуха, количество осадков и солнечных дней, мы сможем предсказывать риски ДО взятия пробы. Например: «после жаркой недели с осадками риск нарушения на озере Х возрастает на 40%».

In [ ]:
# ── Географический анализ ─────────────────────────────────────────────────────
# Эстония разделена на 15 уездов (maakond).
# В opendata XML поле county часто пустое — тогда графики по уездам пропускаются.

_df_c = df.dropna(subset=['county'])
_df_c = _df_c[_df_c['county'].astype(str).str.strip().ne('')]

if len(_df_c) == 0:
    print('Нет заполненного county в данных — графики по уездам недоступны (типично для opendata).')
else:
    county_stats = _df_c.groupby('county')['compliant'].agg(
        total='count',
        violations=lambda x: (x == 0).sum()
    ).reset_index()
    county_stats['violation_rate'] = county_stats['violations'] / county_stats['total'] * 100
    county_stats = county_stats.sort_values('violation_rate', ascending=True)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    county_stats.plot(
        kind='barh', x='county', y='violation_rate', ax=axes[0],
        color='#e07b7b', edgecolor='white'
    )
    axes[0].axvline(x=county_stats['violation_rate'].mean(), color='gray',
                    linestyle='--', label='Среднее')
    axes[0].set_title('% нарушений по уездам')
    axes[0].set_xlabel('% нарушений')
    axes[0].set_ylabel('Уезд')
    axes[0].legend()

    county_top = county_stats.sort_values('total', ascending=False).head(10)
    axes[1].barh(county_top['county'], county_top['total'], color='#c8d8e8',
                 edgecolor='white', label='Всего проб')
    axes[1].barh(county_top['county'], county_top['violations'], color='#e07b7b',
                 edgecolor='white', label='Нарушения')
    axes[1].set_title('Топ-10 уездов: объём проб и нарушения')
    axes[1].set_xlabel('Количество проб')
    axes[1].legend()

    plt.suptitle('Географический анализ нарушений', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print('\nТоп-5 уездов по % нарушений:')
    print(county_stats.nlargest(5, 'violation_rate')[['county', 'total', 'violations', 'violation_rate']].to_string(index=False))


In [ ]:
# ── Распределение числовых параметров: норма vs нарушение ────────────────────
# KDE (Kernel Density Estimate) — оценка плотности распределения.
# Это «сглаженная» гистограмма: показывает, как распределены значения.
# Если кривые для класса 0 и класса 1 сильно разделены —
# признак хорошо разделяет классы и будет полезен для модели.

numeric_cols = ['e_coli', 'enterococci', 'ph', 'transparency']
available_cols = [c for c in numeric_cols if c in df.columns and df[c].notna().sum() > 10]

if not available_cols:
    print('Числовые параметры не найдены в данных — проверьте загрузку.')
else:
    n_cols = len(available_cols)
    fig, axes = plt.subplots(2, n_cols, figsize=(5 * n_cols, 8))
    if n_cols == 1:
        axes = axes.reshape(2, 1)

    for i, col in enumerate(available_cols):
        col_data = df[[col, 'compliant']].dropna()
        norm = col_data[col_data['compliant'] == 1][col]
        viol = col_data[col_data['compliant'] == 0][col]

        # Гистограмма (логарифмическая шкала для E.coli / энтерококков)
        norm.plot.hist(bins=30, ax=axes[0][i], alpha=0.6, color='#5b9bd5',
                       label='Норма (1)', edgecolor='white')
        viol.plot.hist(bins=30, ax=axes[0][i], alpha=0.6, color='#e07b7b',
                       label='Нарушение (0)', edgecolor='white')
        axes[0][i].set_title(f'{col}: гистограмма')
        axes[0][i].set_xlabel('Значение')
        axes[0][i].legend(fontsize=9)

        # Box plot
        col_data.boxplot(column=col, by='compliant', ax=axes[1][i],
                         boxprops=dict(color='#333'),
                         medianprops=dict(color='red', linewidth=2))
        axes[1][i].set_title(f'{col}: box plot')
        axes[1][i].set_xlabel('compliant (0=нарушение, 1=норма)')
        plt.sca(axes[1][i])
        plt.title(f'{col}: медиана и разброс')

    plt.suptitle('Распределение параметров: Норма vs Нарушение', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print('\nМедианные значения по классам:')
    for col in available_cols:
        col_data = df[[col, 'compliant']].dropna()
        med_norm = col_data[col_data['compliant'] == 1][col].median()
        med_viol = col_data[col_data['compliant'] == 0][col].median()
        print(f'  {col:15s}: норма={med_norm:.2f}, нарушение={med_viol:.2f}')

### Как читать box plot (ящик с усами)?

```
          ┌───┬───┐
   ──── ○ │   │ ● │ ───── ○   ○
          └───┴───┘
   ↑       ↑   ↑   ↑
   выброс  Q1  Q2  Q3  выброс
          (медиана)
```

- **Q1** (25-й процентиль): 25% значений ниже этой точки
- **Q2 / медиана** (50-й процентиль): половина значений выше, половина ниже
- **Q3** (75-й процентиль): 75% значений ниже этой точки
- **IQR** = Q3 − Q1 = межквартильный размах (интерквартильный диапазон)
- **Усы**: обычно до 1.5 × IQR от Q1 и Q3
- **Кружки за усами**: выбросы (outliers) — значения, сильно отличающиеся от большинства

Если ящики для класса 0 и класса 1 не перекрываются — признак **хорошо разделяет классы**.

In [ ]:
# ── Корреляционная матрица ────────────────────────────────────────────────────
# Корреляция Пирсона — мера линейной связи между двумя признаками.
# Значения от -1 до +1:
#   +1 = идеальная прямая зависимость (растут вместе)
#   0  = нет линейной связи
#  -1  = идеальная обратная зависимость (один растёт, другой падает)
#
# Для модели опасна высокая корреляция МЕЖДУ признаками (мультиколлинеарность):
# если два признака сильно коррелируют, один из них может быть лишним.

corr_cols = [c for c in ['e_coli', 'enterococci', 'ph', 'transparency', 'compliant']
             if c in df.columns]
corr_data = df[corr_cols].dropna()

if len(corr_data) < 2:
    print('Недостаточно строк без пропусков для корреляции — пропуск heatmap.')
else:
    corr_matrix = corr_data.corr()
    if corr_matrix.isna().all().all():
        print('Матрица корреляций не определена (все NaN) — пропуск графика.')
    else:
        fig, ax = plt.subplots(figsize=(7, 6))
        sns.heatmap(
            corr_matrix,
            annot=True,
            fmt='.2f',
            cmap='RdBu_r',
            center=0,
            square=True,
            ax=ax,
            linewidths=0.5,
            annot_kws={'size': 11}
        )
        ax.set_title('Матрица корреляций (Пирсон)', fontsize=13)
        plt.tight_layout()
        plt.show()

    if 'compliant' in corr_matrix.columns:
        print('\nКорреляция признаков с целевой переменной (compliant):')
        corr_with_target = (
            corr_matrix['compliant'].drop('compliant').sort_values(key=abs, ascending=False)
        )
        corr_with_target = corr_with_target.dropna()
        if len(corr_with_target) == 0:
            print('  (нет конечных значений корреляции с compliant)')
        else:
            for feat, val in corr_with_target.items():
                bar = '█' * int(abs(val) * 20)
                direction = '+' if val > 0 else '-'
                print(f'  {feat:15s}: {val:+.3f}  {direction}{bar}')


---
## 8. Инженерия признаков (Feature Engineering)

**Feature Engineering** — процесс создания новых признаков из имеющихся данных. Хорошие признаки часто важнее, чем сложные алгоритмы.

### Какие признаки мы создаём и зачем?

**1. Отношение к нормативу (`*_ratio`)**

Вместо сырого значения E.coli = 450 создаём:
```python
e_coli_ratio = e_coli / 500  # норма = 500 КОЕ/100 мл
# Результат: 0.9 → 90% от нормы, ещё в пределах
```
Если `ratio > 1.0` → нарушение норматива. Этот признак **нормализует** значения разных параметров и явно кодирует нарушение.

**2. Временные признаки**
- `month` — месяц (1–12): поможет поймать сезонность
- `year` — год: поможет поймать тренды
- `is_summer` — бинарный флаг (июнь/июль/август = 1)

**3. Индикаторы пропусков (`*_missing`)**
- Для каждого параметра: `e_coli_missing = 1` если значение отсутствует, иначе `0`
- Пропуск «измерение не проводилось» — тоже информация

**4. Кодирование категорий**
- `county` (уезд) — строка. Модель работает только с числами.
  - Label Encoding: Harju = 0, Tartu = 1, Pärnu = 2, ...
  - One-Hot Encoding: создаём столбцы `county_Harju`, `county_Tartu`, ... (1 если этот уезд, 0 иначе)
- `domain` (тип воды) → One-Hot
- `season` (сезон) → One-Hot

In [ ]:
# build_dataset() выполняет весь пайплайн инженерии признаков:
# 1. Удаляет строки без целевой переменной
# 2. Добавляет временные признаки
# 3. Добавляет ratio-признаки (отношение к нормативу)
# 4. Добавляет индикаторы пропусков
# 5. Кодирует категориальные признаки
# Возвращает (X, y) — матрицу признаков и вектор целевой переменной

X, y = build_dataset(df)

print(f'Матрица признаков X: {X.shape[0]} строк × {X.shape[1]} признаков')
print(f'Целевая переменная y: {len(y)} значений')
print(f'\nСписок признаков:')
for i, col in enumerate(X.columns):
    print(f'  {i+1:2d}. {col}')

---
## 9. Разделение на обучающую и тестовую выборки

**Почему нельзя обучать и тестировать на одних данных?**

Представьте, что вы дали студенту задание выучить ответы на экзамен, а потом спрашиваете именно эти вопросы. Студент ответит отлично, но мы ничего не узнаем о реальном понимании материала.

Аналогично с моделью: если обучить и оценить на одних данных, модель просто «запомнит» ответы (переобучение).

**Схема разделения:**
```
80% → Train set  →  Модель учится
20% → Test set   →  Проверяем на данных, которые модель не видела
```

**`stratify=y`** — стратифицированное разделение. Гарантирует, что в train и test одинаковая пропорция классов. Без этого можно случайно получить test set, в котором почти нет нарушений.

**`random_state=42`** — фиксируем случайность для воспроизводимости. «42» — дань уважения «Автостопом по галактике», это принято в ML-сообществе.

**Кросс-валидация (Cross-Validation, CV)** — более продвинутый подход. Разбиваем данные на K частей (folds), обучаем K раз (каждый раз одна часть — тест, остальные — обучение). Более надёжная оценка. Мы будем использовать CV при подборе гиперпараметров.

In [ ]:
from sklearn.model_selection import train_test_split

# test_size=0.2 → 20% данных идёт в тест
# stratify=y → сохраняем пропорцию классов
# random_state=42 → фиксируем случайность
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(f'Train set: {len(X_train)} проб  ({len(X_train)/len(X)*100:.0f}%)')
print(f'Test set:  {len(X_test)} проб  ({len(X_test)/len(X)*100:.0f}%)')

print(f'\nРаспределение классов в Train:')
print(f'  Норма: {(y_train==1).sum()} ({(y_train==1).mean()*100:.1f}%)')
print(f'  Нарушение: {(y_train==0).sum()} ({(y_train==0).mean()*100:.1f}%)')

print(f'\nРаспределение классов в Test:')
print(f'  Норма: {(y_test==1).sum()} ({(y_test==1).mean()*100:.1f}%)')
print(f'  Нарушение: {(y_test==0).sum()} ({(y_test==0).mean()*100:.1f}%)')

---
## 10. Предобработка: заполнение пропусков и масштабирование

### Заполнение пропусков (Imputation)

Большинство ML-алгоритмов не умеют работать с NaN. Нужно их заполнить.

**Стратегии:**
- `mean` (среднее) — плохо при выбросах: одно огромное значение E.coli сдвинет среднее
- `median` (медиана) — **наш выбор**: устойчива к выбросам, лучше для данных с хвостами
- `most_frequent` (мода) — для категориальных признаков
- `constant` (константа) — заполнить нулём или спецзначением

**Критическое правило:** `fit` только на train!
```python
imputer.fit(X_train)      # модель запоминает медиану по ОБУЧАЮЩИМ данным
imputer.transform(X_train)  # применяем к train
imputer.transform(X_test)   # применяем ТЕ ЖЕ медианы к test
```
Если мы fit на всём датасете — мы «утекаем» информацию из test в train. Это называется **data leakage** и приводит к завышенным оценкам качества модели.

### Масштабирование (Scaling)

Пробл: признаки имеют разный масштаб:
- E.coli: от 0 до 10000+
- pH: от 0 до 14
- is_summer: только 0 или 1

**StandardScaler** (стандартизация) приводит каждый признак к среднему = 0, стандартному отклонению = 1:
```
X_scaled = (X - mean) / std
```

Это важно для **логистической регрессии** (и других линейных моделей), которые чувствительны к масштабу. Random Forest — нет, но масштабирование и ему не навредит.

In [ ]:
# impute_and_scale() — наш готовый пайплайн из src/features.py:
# 1. SimpleImputer(strategy='median') — заполняет NaN медианами
# 2. StandardScaler() — стандартизирует признаки
# Важно: fit только на X_train!

X_train_scaled, X_test_scaled = impute_and_scale(X_train, X_test)

print('Масштабирование выполнено.')
print(f'X_train_scaled: {X_train_scaled.shape}')
print(f'X_test_scaled:  {X_test_scaled.shape}')
print(f'\nПроверка: пропусков в X_train_scaled: {X_train_scaled.isna().sum().sum()}')
print(f'Проверка: пропусков в X_test_scaled:  {X_test_scaled.isna().sum().sum()}')

# После стандартизации: смотрим на несколько признаков
print(f'\nСтатистика после масштабирования (должно быть ~mean=0, std=1):')
for col in list(X_train_scaled.columns[:4]):
    print(f'  {col:20s}: mean={X_train_scaled[col].mean():.3f}, std={X_train_scaled[col].std():.3f}')

---
## 11. Модель 1: Логистическая регрессия (Logistic Regression)

### Что это?

**Логистическая регрессия** — несмотря на слово «регрессия» в названии, это метод **классификации**. Исторически так сложилось.

Идея: линейная комбинация признаков + сигмоид-функция → вероятность класса 1.

```
z = w₀ + w₁·x₁ + w₂·x₂ + ... + wₙ·xₙ
p(class=1) = sigmoid(z) = 1 / (1 + e^(-z))
```

Если `p > 0.5` → предсказываем класс 1 (норма)  
Если `p ≤ 0.5` → предсказываем класс 0 (нарушение)

### Почему используем как baseline?

**Baseline** — простейшая разумная модель, с которой сравниваем всё остальное.

Логистическая регрессия:
- Быстрая
- Интерпретируемая (коэффициенты показывают влияние каждого признака)
- Хорошая точка отсчёта

### Ключевые параметры:

- **`class_weight='balanced'`** — автоматически увеличивает вес редкого класса (нарушений). sklearn считает: `weight = n_samples / (n_classes × n_samples_per_class)`
- **`max_iter=1000`** — максимум итераций оптимизации. Увеличиваем, чтобы алгоритм успел сойтись.
- **`C`** — параметр регуляризации. Маленький C = сильная регуляризация = более простая модель. Большой C = слабая регуляризация = модель сложнее и может переобучиться.
- **`random_state=42`** — воспроизводимость

In [ ]:
from sklearn.linear_model import LogisticRegression

# Создаём и обучаем логистическую регрессию
lr_model = LogisticRegression(
    class_weight='balanced',  # учитываем дисбаланс классов
    max_iter=1000,            # даём достаточно итераций
    C=1.0,                    # регуляризация (стандартное значение)
    random_state=42,
    solver='lbfgs'            # алгоритм оптимизации (хорош для средних датасетов)
)

lr_model.fit(X_train_scaled, y_train)

print('Логистическая регрессия обучена!')
print(f'Коэффициентов (весов): {len(lr_model.coef_[0])}')

# Смотрим на топ-5 наиболее «влиятельных» признаков
coef_df = pd.DataFrame({
    'feature': X_train_scaled.columns,
    'coefficient': lr_model.coef_[0]
}).sort_values('coefficient', key=abs, ascending=False)

print('\nТоп-10 признаков по |коэффициенту|:')
print('(отрицательный коэффициент → признак указывает на нарушение)')
for _, row in coef_df.head(10).iterrows():
    bar = '█' * int(abs(row['coefficient']) * 5)
    sign = '+' if row['coefficient'] > 0 else '-'
    print(f"  {row['feature']:25s}: {row['coefficient']:+.3f}  {sign}{bar}")

---
## 12. Метрики оценки: как понять, хороша ли модель?

Это один из самых важных разделов. Выбор метрики = выбор того, что для нас важно.

### Confusion Matrix (Матрица ошибок)

```
                   Предсказание модели
                   0 (нарушение)   1 (норма)
  Факт:  0  ┌─────────────────┬──────────────┐
 (нарушение)│  TN (True Neg)  │ FP (False P) │
             ├─────────────────┼──────────────┤
  Факт:  1  │  FN (False Neg) │ TP (True P)  │
  (норма)   └─────────────────┴──────────────┘
```

- **TP** (True Positive) — правильно нашли норму
- **TN** (True Negative) — правильно нашли нарушение
- **FP** (False Positive) — ложная тревога: норма → сказали «нарушение»
- **FN** (False Negative) — пропущенная опасность: нарушение → сказали «норма» ⚠️

### Основные метрики

| Метрика | Формула | Что измеряет |
|---------|---------|-------------|
| **Accuracy** | (TP+TN) / всего | Доля правильных ответов (плохо при дисбалансе!) |
| **Precision** | TP / (TP+FP) | Из предсказанных «нарушений», сколько реальных? |
| **Recall** | TP / (TP+FN) | Из реальных нарушений, сколько нашли? |
| **F1-score** | 2·P·R / (P+R) | Гармоническое среднее Precision и Recall |
| **ROC-AUC** | — | Площадь под ROC-кривой (0.5 = случайно, 1.0 = идеально) |

### Наш приоритет: Recall по классу 0 (нарушения)

```
Recall(нарушение) = TN / (TN + FP)
```

«Из всех реальных нарушений — сколько модель нашла?»

Мы хотим этот Recall как можно выше. Пропустить опасную воду хуже, чем излишне закрыть пляж.

### ROC-AUC

**ROC (Receiver Operating Characteristic)** — кривая, показывающая компромисс между TPR и FPR при разных порогах классификации.  
**AUC (Area Under Curve)** — площадь под этой кривой.
- AUC = 0.5 → модель не лучше монетки
- AUC = 0.7 → приемлемо
- AUC = 0.8–0.9 → хорошо
- AUC = 1.0 → идеально (подозрительно!)

In [ ]:
# Оцениваем логистическую регрессию
lr_results = evaluate_model(lr_model, X_test_scaled, y_test, 'Logistic Regression')
lr_results['y_test'] = y_test  # добавляем для plot_roc_curve

# Confusion Matrix
plot_confusion_matrix(y_test, lr_results['y_pred'], 'Logistic Regression')

---
## 13. Модель 2: Случайный лес (Random Forest)

### Что такое Decision Tree (Дерево решений)?

Дерево решений — модель, которая задаёт последовательные вопросы-условия:
```
E.coli > 300?
├── ДА: enterococci > 100?
│       ├── ДА → Нарушение (0)
│       └── НЕТ → Норма (1)
└── НЕТ: pH < 6.5?
        ├── ДА → Нарушение (0)
        └── НЕТ → Норма (1)
```

Одно дерево — нестабильно: маленькое изменение в данных даёт совсем другое дерево.

### Random Forest — ансамбль деревьев

**Ансамблевый метод (Ensemble)** — объединяет предсказания многих слабых моделей в одно сильное.

**Random Forest** строит N деревьев (например, 100), каждое на:
- Случайной подвыборке данных (bootstrap — случайная выборка с возвращением)
- Случайном подмножестве признаков (для разнообразия)

Предсказание = голосование: большинство деревьев говорит «норма» → результат «норма».

### Преимущества Random Forest:
- Не нужно масштабирование признаков
- Устойчив к выбросам и шуму
- Даёт **feature importance** — важность каждого признака
- Редко переобучается
- Хорошо работает «из коробки»

### Ключевые гиперпараметры:
- **`n_estimators`** — количество деревьев. Больше = лучше, но медленнее. 100–200 — разумно.
- **`max_depth`** — максимальная глубина дерева. Ограничивает сложность, помогает против переобучения.
- **`min_samples_leaf`** — минимум примеров в листе. Больше = более «осторожные» деревья.
- **`class_weight='balanced'`** — учитываем дисбаланс классов.
- **`n_jobs=-1`** — использовать все ядра процессора (параллелизм).

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Random Forest не требует масштабирования,
# но для унификации используем X_train_scaled (вреда нет)
rf_model = RandomForestClassifier(
    n_estimators=100,          # 100 деревьев
    max_depth=None,            # деревья растут до конца (контролируется min_samples_leaf)
    min_samples_leaf=5,        # в каждом листе минимум 5 проб (защита от переобучения)
    class_weight='balanced',   # учитываем дисбаланс классов
    random_state=42,
    n_jobs=-1                  # параллельное обучение на всех ядрах CPU
)

rf_model.fit(X_train_scaled, y_train)
print('Random Forest обучен!')
print(f'Количество деревьев: {rf_model.n_estimators}')

# Кросс-валидация (CV) на обучающих данных
# cv=5 → 5-fold CV: делим train на 5 частей, 5 раз обучаем/тестируем
# scoring='f1' → оцениваем по F1-score
cv_scores = cross_val_score(rf_model, X_train_scaled, y_train, cv=5, scoring='f1')
print(f'\nКросс-валидация (F1, cv=5):')
print(f'  Scores: {[f"{s:.3f}" for s in cv_scores]}')
print(f'  Mean: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')

In [ ]:
# Оцениваем Random Forest
rf_results = evaluate_model(rf_model, X_test_scaled, y_test, 'Random Forest')
rf_results['y_test'] = y_test

# Confusion Matrix
plot_confusion_matrix(y_test, rf_results['y_pred'], 'Random Forest')

---
## 14. Сравнение моделей

Сравниваем обе модели по всем метрикам. Помним: **главный приоритет — Recall по классу 0 (нарушения)**.

In [ ]:
# Сравнительная таблица
comparison = compare_models([lr_results, rf_results])
print('Сравнение моделей по метрикам:')
print(comparison.to_string())

# Визуализация сравнения
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Метрики по барам
metrics = ['ROC-AUC', 'F1 (нарушение)', 'Recall (нарушение)', 'Precision (нарушение)']
if all(m in comparison.columns for m in metrics):
    x = range(len(metrics))
    w = 0.35
    axes[0].bar([xi - w/2 for xi in x], comparison.loc['Logistic Regression', metrics], w,
                label='Logistic Regression', color='#5b9bd5', edgecolor='white')
    axes[0].bar([xi + w/2 for xi in x], comparison.loc['Random Forest', metrics], w,
                label='Random Forest', color='#70ad47', edgecolor='white')
    axes[0].set_xticks(list(x))
    axes[0].set_xticklabels(metrics, rotation=20, ha='right')
    axes[0].set_ylim(0, 1.1)
    axes[0].set_title('Сравнение метрик')
    axes[0].legend()
    axes[0].axhline(y=0.5, color='red', linestyle=':', linewidth=1, label='Случайный baseline')

# ROC-кривые
from sklearn.metrics import roc_curve
for result, color, name in [
    (lr_results, '#5b9bd5', 'LR'),
    (rf_results, '#70ad47', 'RF')
]:
    fpr, tpr, _ = roc_curve(result['y_test'], result['y_prob'])
    axes[1].plot(fpr, tpr, color=color, linewidth=2,
                 label=f"{name} (AUC={result['roc_auc']:.3f})")

axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Случайный baseline (AUC=0.5)')
axes[1].set_xlabel('False Positive Rate (FPR)')
axes[1].set_ylabel('True Positive Rate (TPR = Recall)')
axes[1].set_title('ROC-кривые')
axes[1].legend()

plt.suptitle('Сравнение моделей: Logistic Regression vs Random Forest', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 15. Feature Importance: что влияет на качество воды?

**Feature Importance** в Random Forest — мера того, насколько каждый признак помогает разделять классы. Вычисляется на основе того, насколько каждый признак уменьшает «нечистоту» (impurity) при разбиении данных в деревьях.

Это один из самых практически ценных результатов:
- Понимаем, какие параметры критичны для безопасности воды
- Можем рекомендовать, что измерять в первую очередь
- Можем упростить модель, убрав бесполезные признаки

In [ ]:
# Feature Importance для Random Forest
plot_feature_importance(rf_model, list(X_train_scaled.columns), top_n=15)

# Выводим числовые значения
importances = pd.Series(rf_model.feature_importances_,
                         index=X_train_scaled.columns).sort_values(ascending=False)

print('Топ-15 признаков по важности:')
for feat, imp in importances.head(15).items():
    bar = '█' * int(imp * 300)
    print(f'  {feat:25s}: {imp:.4f}  {bar}')

---
## 16. Будущее проекта: внешние данные и более умные модели

Текущая версия работает только с лабораторными данными — результатами, которые уже есть. Это **ретроспективный анализ**: мы объясняем прошлое.

Настоящая ценность ML — **проспективное предсказание**: предупреждать ДО того, как проблема произошла.

---

### 16.1 Погодные данные

Откуда брать:
- **Open-Meteo API** — бесплатный, без регистрации, исторические данные с 1940 года
- **ERA5 (Copernicus)** — европейский климатический реанализ
- **Estonian Weather Service (EMH)** — данные метеостанций Эстонии

Что добавить:

| Признак | Почему важен |
|---------|-------------|
| `temp_max_7d` | Температура воды коррелирует с воздухом. Жара → быстрее размножаются бактерии |
| `precip_sum_7d` | Осадки смывают загрязнения в водоёмы (навоз, нитраты) |
| `wind_speed` | Перемешивание воды влияет на концентрацию бактерий и водорослей |
| `solar_radiation` | UV-излучение убивает бактерии. Мало солнца → больше риск |
| `days_since_rain` | После долгой засухи и потом дождя — высокий смыв |

```python
import openmeteo_requests  # pip install openmeteo-requests

# Пример: температура на Пирита за купальный сезон 2024
url = 'https://archive-api.open-meteo.com/v1/archive'
params = {
    'latitude': 59.47,   # Pirita, Tallinn
    'longitude': 24.83,
    'start_date': '2024-06-01',
    'end_date': '2024-08-31',
    'daily': ['temperature_2m_max', 'precipitation_sum', 'wind_speed_10m_max']
}
# Объединяем с пробами воды по дате и месту
```

---

### 16.2 Спутниковые данные

Цветение водорослей (cyanobacteria bloom) видно из космоса! Балтийское море страдает от массового цветения каждое лето.

- **Copernicus Marine Service (CMEMS)** — хлорофилл-а, температура воды, цветность
- **Sentinel-2 / Sentinel-3** (ESA) — оптические снимки, NDVI-подобные индексы воды
- **NASA Earthdata** — MODIS, температура поверхности воды (SST)

```python
# Индекс цианобактерий из Sentinel-3 OLCI
# CYA = f(band8 / band11)  -- упрощённо
# Значение > 20 000 кл/мл → запрет купания в Эстонии
```

---

### 16.3 Временные ряды (Time Series)

Сейчас каждая проба — независимый пример. Но вода в одном месте имеет **историю**.

**Идея:** добавить признаки «состояния водоёма за последние N недель»:
- `e_coli_mean_4w` — среднее E.coli за 4 недели до этой пробы
- `violations_count_30d` — количество нарушений в этом месте за 30 дней
- `last_violation_days` — дней с последнего нарушения

**Продвинутые модели для временных рядов:**
- **LSTM** (Long Short-Term Memory) — нейронная сеть, хорошо запоминает долгосрочные зависимости
- **Prophet** (Facebook/Meta) — предсказание временных рядов с сезонностью
- **XGBoost / LightGBM** с lag-признаками — часто лучше LSTM на небольших данных

---

### 16.4 Географические данные

| Источник данных | Что добавить |
|----------------|--------------|
| OpenStreetMap | Расстояние до ближайшей фермы / дороги / сточных труб |
| Estonian Land Board | Тип почвы вокруг водоёма |
| Estonian Environment Agency | Данные о реках выше по течению |
| Google Maps API | Количество посетителей пляжа (proxy через traffic) |

---

### 16.5 Архитектура будущей системы

```
┌─────────────────────────────────────────────────────────────┐
│                    ИСТОЧНИКИ ДАННЫХ                         │
├──────────────┬──────────────┬──────────────┬───────────────┤
│  Terviseamet │  Open-Meteo  │  Copernicus  │  OSM / Land   │
│  (лаб. анализ)│  (погода)   │  (спутник)   │  Board (геоInformation)  │
└──────┬───────┴──────┬───────┴──────┬───────┴───────┬───────┘
       │              │              │               │
       └──────────────┴──────────────┴───────────────┘
                              │
                    ┌─────────▼─────────┐
                    │  Feature Store    │
                    │  (объединённые    │
                    │   признаки)       │
                    └─────────┬─────────┘
                              │
                    ┌─────────▼─────────┐
                    │  ML Model         │
                    │  (LightGBM / RF)  │
                    └─────────┬─────────┘
                              │
                    ┌─────────▼─────────┐
                    │  API / Dashboard  │
                    │  "Безопасно ли    │
                    │  сейчас на Пирите?"│
                    └───────────────────┘
```

In [ ]:
# Демонстрация: как могли бы выглядеть данные с погодой
# (это учебный пример — реальную загрузку реализуем в следующих ноутбуках)

import numpy as np

# Пример: генерируем реалистичные синтетические записи
np.random.seed(42)
n = 100

example_df = pd.DataFrame({
    'location': np.random.choice(['Pirita', 'Pärnu rand', 'Narva-Jõesuu', 'Pühajärv'], n),
    'e_coli': np.random.exponential(scale=150, size=n).round(),
    'enterococci': np.random.exponential(scale=60, size=n).round(),
    'ph': np.random.normal(7.5, 0.5, n).round(1),
    # --- Будущие погодные признаки ---
    'temp_max_7d': np.random.normal(22, 5, n).round(1),       # °C
    'precip_sum_7d': np.random.exponential(scale=10, size=n).round(1),  # мм
    'wind_speed_avg': np.random.normal(4, 2, n).clip(0).round(1),  # м/с
    'solar_rad_7d': np.random.normal(180, 50, n).clip(0).round(),  # Вт/м²
    # --- Будущие исторические признаки ---
    'violations_30d': np.random.poisson(lam=0.5, size=n),           # количество нарушений
    'days_since_last_violation': np.random.exponential(30, n).round().astype(int),
})

print('Пример будущего датасета с погодными и историческими признаками:')
print(example_df.head(8).to_string(index=False))
print(f'\nВсего признаков в расширенном датасете: {example_df.shape[1]}')
print('\nСравнение: текущих признаков vs расширенных:')
print(f'  Текущих (лаборатория):  {X.shape[1]}')
print(f'  С погодой и историей:   {X.shape[1] + 6} (примерная оценка)')

---
## 17. Итог и следующие шаги

### Что мы сделали в этом ноутбуке:

| Этап | Что узнали |
|------|------------|
| Данные | XML из Terviseamet → pandas DataFrame |
| EDA | Сезонность, географию, распределение параметров |
| Целевая переменная | `compliant`: `vastavus=jah` → 1, `ei` → 0 |
| Feature Engineering | Ratio-признаки, временные, индикаторы пропусков |
| Preprocessing | Median Imputer + StandardScaler (только fit на train!) |
| Модель 1 | Logistic Regression — baseline |
| Модель 2 | Random Forest — основная модель |
| Метрики | Приоритет: Recall по классу 0 (нарушения) |
| Feature Importance | Какие параметры наиболее значимы |

---

### Следующие ноутбуки:

| Файл | Цель |
|------|------|
| `01_eda_supluskoha.ipynb` | Углублённый EDA: только купальные места |
| `02_eda_full.ipynb` | EDA всех 5 доменов |
| `03_preprocessing.ipynb` | Детальная предобработка, Grid Search |
| `04_models.ipynb` | Подбор гиперпараметров, XGBoost, LightGBM |
| `05_evaluation.ipynb` | Финальная оценка, интерпретация, отчёт |

---

### Открытые вопросы (исследовательская программа):

1. **Насколько хуже/лучше погодные признаки** (когда добавим Open-Meteo)?
2. **Можно ли предсказывать за 3–7 дней** только по погодным данным?
3. **Как работает модель по уездам?** Одна модель для всех или отдельные?
4. **Порог классификации**: стоит ли снизить с 0.5 до 0.3, чтобы быть консервативнее?
5. **SMOTE**: поможет ли синтетическое увеличение редкого класса?

---

### Глоссарий ключевых терминов (быстрая шпаргалка):

| Термин | Определение |
|--------|-------------|
| **ML** | Machine Learning — машинное обучение |
| **EDA** | Exploratory Data Analysis — разведочный анализ данных |
| **Feature** | Признак — входная переменная модели |
| **Target** | Целевая переменная — то, что предсказываем |
| **NaN** | Not a Number — маркер пропущенного значения |
| **Overfitting** | Переобучение — модель хорошо на train, плохо на test |
| **Precision** | Точность — доля верных «нарушений» среди всех предсказанных |
| **Recall** | Полнота — доля найденных реальных нарушений |
| **F1-score** | Гармоническое среднее Precision и Recall |
| **ROC-AUC** | Площадь под ROC-кривой — общее качество классификатора |
| **FN** | False Negative — пропущенное нарушение (купаемся в E.coli) |
| **FP** | False Positive — ложная тревога (закрыли чистый пляж) |
| **CV** | Cross-Validation — кросс-валидация |
| **Imputation** | Заполнение пропущенных значений |
| **StandardScaler** | Стандартизация: приводим к mean=0, std=1 |
| **Random Forest** | Ансамбль деревьев решений |
| **КОЕ** | Колониеобразующие единицы — мера количества бактерий |
| **NTU** | Nephelometric Turbidity Units — единица мутности воды |
| **vastavus** | Эст. "соответствие" — поле XML: jah/ei |
| **compliant** | Целевая переменная: 1 = норма, 0 = нарушение |

> **Помните о цене ошибки:** False Negative — человек купается в опасной воде. Это не просто цифра в метрике — это реальный риск для здоровья. Именно поэтому мы оптимизируем Recall, а не только Accuracy.